# Audit re-derivation, crop coverage, and exclusion sensitivity

Runs on the TS-SatFire dataset only. No checkpoints needed. No GPU needed.
Expected runtime: 15-30 minutes.

This answers the following reviewer points directly:
  - Reviewer 2: the 18 vs 21 excluded-training-fires inconsistency
  - Reviewer 3: "summarise original and final usable counts in a single table"
  - Reviewer 3: "clarify whether the 256x256 centre crop may exclude part of the wildfire"
  - Reviewer 3: "a sensitivity analysis showing how different exclusion thresholds
                 affect the final results"
  - Reviewer 3: "the description of the auxiliary FirePred variables is limited"

Everything written here goes straight into the revised Section 3 and Section 4.


In [1]:
import os, glob, json, warnings
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

warnings.filterwarnings("ignore")

try:
    import rasterio
    HAS_RASTERIO = True
except Exception:
    import tifffile
    HAS_RASTERIO = False

OUT = "/kaggle/working"
os.makedirs(OUT, exist_ok=True)
CROP = 256


def find_data_root():
    for p in ["/kaggle/input/datasets/z789456sx/ts-satfire/ts-satfire",
              "/kaggle/input/ts-satfire/ts-satfire/ts-satfire",
              "/kaggle/input/ts-satfire/ts-satfire",
              "/kaggle/input/ts-satfire"]:
        if os.path.isdir(p):
            return p
    for root, dirs, _ in os.walk("/kaggle/input"):
        if any(d.endswith("_fire") for d in dirs):
            return root
        if root.replace("/kaggle/input", "").count(os.sep) >= 5:
            dirs.clear()
    return None


DATA_ROOT = find_data_root()
assert DATA_ROOT, "TS-SatFire dataset not found"
print("DATA_ROOT:", DATA_ROOT)

ALL_DIRS = sorted(d for d in os.listdir(DATA_ROOT)
                  if os.path.isdir(os.path.join(DATA_ROOT, d)))
print("Fire directories found:", len(ALL_DIRS))


DATA_ROOT: /kaggle/input/datasets/z789456sx/ts-satfire/ts-satfire
Fire directories found: 192


## 1. Split definitions

The AF/BA validation set (13 fires) is the FP validation list (15 IDs) minus the
two fires the dataset paper adds for FP only. Training is every remaining
numeric ID. This is stated explicitly so the revised Section 3.2 can cite it.


In [2]:
AF_TEST = [
    "elephant_hill_fire", "eagle_bluff_fire", "double_creek_fire", "sparks_lake_fire",
    "lytton_fire", "chuckegg_creek_fire", "swedish_fire", "sydney_fire",
    "thomas_fire", "tubbs_fire", "carr_fire", "camp_fire",
    "creek_fire", "blue_ridge_fire", "dixie_fire", "mosquito_fire", "calfcanyon_fire",
]

FP_VAL_IDS = ["20568194", "20701026", "20562846", "20700973", "24462610",
              "24462788", "24462753", "24103571", "21998313", "21751303",
              "22141596", "21999381", "23301962", "22712904", "22713339"]
FP_ONLY_VAL = ["23301962", "22713339"]
AFBA_VAL = [i for i in FP_VAL_IDS if i not in FP_ONLY_VAL]

BAFP_TEST = sorted(d for d in ALL_DIRS if d.startswith("US_2021"))
NUMERIC = sorted(d for d in ALL_DIRS if d.isdigit())
TRAIN = [d for d in NUMERIC if d not in FP_VAL_IDS]

print("train (numeric, minus val):", len(TRAIN))
print("AF/BA val:", len(AFBA_VAL), " present on disk:",
      sum(1 for f in AFBA_VAL if f in ALL_DIRS))
print("FP val (paper list):", len(FP_VAL_IDS), " present on disk:",
      sum(1 for f in FP_VAL_IDS if f in ALL_DIRS))
print("AF test:", len(AF_TEST), " present on disk:",
      sum(1 for f in AF_TEST if f in ALL_DIRS))
print("BA/FP test:", len(BAFP_TEST))

missing_val = [f for f in FP_VAL_IDS if f not in ALL_DIRS]
print("FP val IDs absent from this release:", missing_val)


train (numeric, minus val): 137
AF/BA val: 13  present on disk: 13
FP val (paper list): 15  present on disk: 14
AF test: 17  present on disk: 17
BA/FP test: 24
FP val IDs absent from this release: ['23301962']


## 2. Per-fire scan

For every day of every fire we read only band 7 (active fire) and band 8
(burned area) rather than the whole raster, which keeps this fast.

Recorded per fire:
  - number of days
  - days with any finite AF label, days with any finite BA label
  - positive-pixel counts inside and outside the 256x256 centre crop
  - whether the fire has a VIIRS_Day directory at all


In [3]:
def read_band(path, idx1):
    """Read one 1-indexed band. Returns None if absent."""
    if HAS_RASTERIO:
        with rasterio.open(path) as src:
            if src.count < idx1:
                return None
            return src.read(idx1).astype(np.float32)
    arr = tifffile.imread(path).astype(np.float32)
    if arr.ndim == 2:
        arr = arr[np.newaxis]
    if arr.shape[0] < idx1:
        return None
    return arr[idx1 - 1]


def crop_split(a, size=CROP):
    """Return (inside_crop, full) views."""
    h, w = a.shape
    r0, c0 = max((h - size) // 2, 0), max((w - size) // 2, 0)
    return a[r0:r0 + size, c0:c0 + size], a


def scan_fire(fid):
    fdir = os.path.join(DATA_ROOT, fid)
    ddir = os.path.join(fdir, "VIIRS_Day")
    rec = {"fire": fid, "has_day_dir": os.path.isdir(ddir), "n_days": 0,
           "af_days_labelled": 0, "ba_days_labelled": 0,
           "af_pos_in": 0, "af_pos_all": 0,
           "ba_pos_in": 0, "ba_pos_all": 0,
           "af_pos_in_ref": 0, "af_pos_all_ref": 0,
           "height": None, "width": None, "error": None}
    if not rec["has_day_dir"]:
        return rec

    files = sorted(glob.glob(os.path.join(ddir, "*.tif")))
    rec["n_days"] = len(files)
    try:
        for f in files:
            b7 = read_band(f, 7)
            b8 = read_band(f, 8)
            if b7 is not None and rec["height"] is None:
                rec["height"], rec["width"] = int(b7.shape[0]), int(b7.shape[1])

            if b7 is not None:
                if np.isfinite(b7).any():
                    rec["af_days_labelled"] += 1
                # manuscript rule: confidence >= 7
                m = np.nan_to_num(b7, nan=0.0) >= 7
                mi, ma = crop_split(m)
                rec["af_pos_in"] += int(mi.sum()); rec["af_pos_all"] += int(ma.sum())
                # reference rule: any nonzero after nan_to_num
                r = np.nan_to_num(b7, nan=0.0) > 0
                ri, ra = crop_split(r)
                rec["af_pos_in_ref"] += int(ri.sum()); rec["af_pos_all_ref"] += int(ra.sum())

            if b8 is not None:
                fin = np.isfinite(b8)
                if fin.any():
                    rec["ba_days_labelled"] += 1
                bi, ba = crop_split(fin)
                rec["ba_pos_in"] += int(bi.sum()); rec["ba_pos_all"] += int(ba.sum())
    except Exception as e:
        rec["error"] = "{}: {}".format(type(e).__name__, e)
    return rec


TARGETS = sorted(set(TRAIN + FP_VAL_IDS + AF_TEST + BAFP_TEST))
TARGETS = [t for t in TARGETS if t in ALL_DIRS]
print("Scanning", len(TARGETS), "fires...")

with ThreadPoolExecutor(max_workers=8) as ex:
    records = list(ex.map(scan_fire, TARGETS))

df = pd.DataFrame(records)
df["af_frac_days"] = np.where(df.n_days > 0, df.af_days_labelled / df.n_days, 0.0)
df["ba_frac_days"] = np.where(df.n_days > 0, df.ba_days_labelled / df.n_days, 0.0)


def split_of(f):
    if f in AF_TEST:
        return "af_test"
    if f in BAFP_TEST:
        return "bafp_test"
    if f in FP_VAL_IDS:
        return "val"
    return "train"


df["split"] = df.fire.map(split_of)
df.to_csv(os.path.join(OUT, "audit_per_fire.csv"), index=False)
print("Scanned. Errors:", int(df.error.notna().sum()))
print(df.groupby("split").size())


Scanning 192 fires...
Scanned. Errors: 0
split
af_test       17
bafp_test     24
train        137
val           14
dtype: int64


## 3. Authoritative exclusion counts

This settles the 18-vs-21 question. Three reasons are reported separately so
the revised Table 1 can state a single definition and show its components.


In [4]:
def classify(row, task, min_frac=0.5):
    if not row["has_day_dir"]:
        return "no_viirs_day_dir"
    col = "af_days_labelled" if task == "af" else "ba_days_labelled"
    frac = row["af_frac_days"] if task == "af" else row["ba_frac_days"]
    if row[col] == 0:
        return "all_nan_label"
    if frac < min_frac:
        return "under_half_days"
    return "usable"


rows = []
for task in ["af", "ba"]:
    for split in ["train", "val", "af_test", "bafp_test"]:
        sub = df[df.split == split]
        if sub.empty:
            continue
        cls = sub.apply(lambda r: classify(r, task), axis=1)
        rows.append({
            "task": task.upper(), "split": split, "total": len(sub),
            "usable": int((cls == "usable").sum()),
            "no_dir": int((cls == "no_viirs_day_dir").sum()),
            "all_nan": int((cls == "all_nan_label").sum()),
            "under_half": int((cls == "under_half_days").sum()),
            "excluded_total": int((cls != "usable").sum()),
        })
summary = pd.DataFrame(rows)
summary.to_csv(os.path.join(OUT, "audit_summary_table.csv"), index=False)
print(summary.to_string(index=False))

print("\nTwo defensible definitions of AF training exclusions:")
tr = df[df.split == "train"]
cls = tr.apply(lambda r: classify(r, "af"), axis=1)
print("  excluded because the label band is unusable (all-NaN or <50% of days):",
      int(((cls == "all_nan_label") | (cls == "under_half_days")).sum()))
print("  plus fires with no VIIRS_Day directory at all:",
      int((cls == "no_viirs_day_dir").sum()))
print("  total unavailable for training:", int((cls != "usable").sum()))
print("\nUse ONE of these in the abstract and Table 1, and name it explicitly.")

for task in ["af", "ba"]:
    cls_all = df.apply(lambda r: classify(r, task), axis=1)
    bad = sorted(df.fire[(cls_all != "usable") & (df.split == "train")].tolist())
    json.dump(bad, open(os.path.join(OUT, "excluded_{}_train.json".format(task)), "w"),
              indent=1)
    print("\n{} train exclusion list written ({} fires)".format(task.upper(), len(bad)))


task     split  total  usable  no_dir  all_nan  under_half  excluded_total
  AF     train    137     117      13        4           3              20
  AF       val     14      12       1        1           0               2
  AF   af_test     17      14       0        2           1               3
  AF bafp_test     24      21       0        0           3               3
  BA     train    137      89      13       18          17              48
  BA       val     14       8       1        5           0               6
  BA   af_test     17       9       0        7           1               8
  BA bafp_test     24      18       0        2           4               6

Two defensible definitions of AF training exclusions:
  excluded because the label band is unusable (all-NaN or <50% of days): 7
  plus fires with no VIIRS_Day directory at all: 13
  total unavailable for training: 20

Use ONE of these in the abstract and Table 1, and name it explicitly.

AF train exclusion list written (2

## 4. Does the 256x256 centre crop cut off part of the fire?

Reviewer 3 asked this directly. Reported as the percentage of labelled positive
pixels that fall outside the crop.


In [5]:
cov = df[df.split.isin(["af_test", "bafp_test"])].copy()
cov["af_outside_pct"] = np.where(cov.af_pos_all > 0,
                                 100 * (1 - cov.af_pos_in / cov.af_pos_all), np.nan)
cov["ba_outside_pct"] = np.where(cov.ba_pos_all > 0,
                                 100 * (1 - cov.ba_pos_in / cov.ba_pos_all), np.nan)
cov["ba_burn_frac_in_crop_pct"] = np.where(
    cov.n_days > 0, 100 * cov.ba_pos_in / (cov.n_days * CROP * CROP), np.nan)

show = cov[["fire", "split", "height", "width", "af_outside_pct",
            "ba_outside_pct", "ba_burn_frac_in_crop_pct"]].sort_values(
    ["split", "ba_outside_pct"], ascending=[True, False])
show.to_csv(os.path.join(OUT, "crop_coverage.csv"), index=False)
print(show.to_string(index=False, float_format=lambda x: "{:.3f}".format(x)))

print("\nAF test: mean labelled pixels outside crop: {:.2f}%".format(
    cov[cov.split == "af_test"].af_outside_pct.mean()))
print("BA/FP test: mean burned pixels outside crop: {:.2f}%".format(
    cov[cov.split == "bafp_test"].ba_outside_pct.mean()))


                         fire     split  height   width  af_outside_pct  ba_outside_pct  ba_burn_frac_in_crop_pct
             eagle_bluff_fire   af_test 596.000 595.000           0.997         100.000                     0.000
            double_creek_fire   af_test 596.000 595.000          70.262          68.103                     8.966
              blue_ridge_fire   af_test 596.000 595.000          31.280          22.044                     1.492
                   creek_fire   af_test 596.000 595.000          19.036          19.780                     8.469
                   tubbs_fire   af_test 596.000 595.000          24.567          16.846                     1.061
                mosquito_fire   af_test 596.000 594.000             NaN           0.996                     1.077
                   dixie_fire   af_test 596.000 595.000           2.830           0.898                     2.728
              calfcanyon_fire   af_test 596.000 594.000             NaN           0.269 

## 5. Exclusion-threshold sensitivity

How many test fires would be excluded at each burn-fraction cut-off, and how
many training fires change status as the usable-days criterion moves.


In [6]:
print("Burn-fraction cut-off applied to the 24 BA/FP test fires")
print("(the manuscript uses 0.001%):\n")
te = cov[cov.split == "bafp_test"]
for thr in [0.0, 0.0001, 0.001, 0.01, 0.1, 1.0]:
    ex = te[te.ba_burn_frac_in_crop_pct < thr]
    print("  cut-off {:>7.4f}%  ->  excluded {:2d} / {:d}".format(
        thr, len(ex), len(te)))

print("\nFires excluded at the manuscript's 0.001% cut-off:")
for f in sorted(te[te.ba_burn_frac_in_crop_pct < 0.001].fire.tolist()):
    print("   ", f)

print("\nUsable-days criterion on the training split:")
for mf in [0.25, 0.50, 0.75]:
    for task in ["af", "ba"]:
        sub = df[df.split == "train"]
        cls = sub.apply(lambda r: classify(r, task, min_frac=mf), axis=1)
        print("  min_frac {:.2f}  {}  usable {:3d}  excluded {:3d}".format(
            mf, task.upper(), int((cls == "usable").sum()),
            int((cls != "usable").sum())))


Burn-fraction cut-off applied to the 24 BA/FP test fires
(the manuscript uses 0.001%):

  cut-off  0.0000%  ->  excluded  0 / 24
  cut-off  0.0001%  ->  excluded  3 / 24
  cut-off  0.0010%  ->  excluded  5 / 24
  cut-off  0.0100%  ->  excluded  5 / 24
  cut-off  0.1000%  ->  excluded  8 / 24
  cut-off  1.0000%  ->  excluded 16 / 24

Fires excluded at the manuscript's 0.001% cut-off:
    US_2021_FL2521008104520210308
    US_2021_NM3323810847220210520
    US_2021_NM3340210587120210426
    US_2021_NM3344410803520210514
    US_2021_NM3676810505920211120

Usable-days criterion on the training split:
  min_frac 0.25  AF  usable 119  excluded  18
  min_frac 0.25  BA  usable  97  excluded  40
  min_frac 0.50  AF  usable 117  excluded  20
  min_frac 0.50  BA  usable  89  excluded  48
  min_frac 0.75  AF  usable 104  excluded  33
  min_frac 0.75  BA  usable  82  excluded  55


## 6. FirePred auxiliary bands

The manuscript drops band 3 because it was all zeros in one probe fire.
Reviewer 3 asked for better documentation of these variables. This checks the
claim across every fire and reports which bands are constant.


In [7]:
def fp_band_stats(fid, max_days=3):
    fdir = os.path.join(DATA_ROOT, fid, "FirePred")
    if not os.path.isdir(fdir):
        return None
    files = sorted(glob.glob(os.path.join(fdir, "*.tif")))[:max_days]
    if not files:
        return None
    out = []
    for f in files:
        try:
            if HAS_RASTERIO:
                with rasterio.open(f) as src:
                    a = src.read().astype(np.float32)
            else:
                a = tifffile.imread(f).astype(np.float32)
            out.append(a)
        except Exception:
            return None
    a = np.stack([x[:19] for x in out if x.shape[0] >= 19], axis=0) if out else None
    if a is None or a.size == 0:
        return None
    a = np.nan_to_num(a, nan=0.0)
    return {"n_bands": int(a.shape[1]),
            "allzero": [bool(np.all(a[:, b] == 0)) for b in range(a.shape[1])],
            "std": [float(np.std(a[:, b])) for b in range(a.shape[1])]}


sample = TRAIN[:60] + BAFP_TEST[:10]
stats = {}
with ThreadPoolExecutor(max_workers=8) as ex:
    for fid, s in zip(sample, ex.map(fp_band_stats, sample)):
        if s:
            stats[fid] = s

if stats:
    nb = max(s["n_bands"] for s in stats.values())
    allzero_count = np.zeros(nb, dtype=int)
    for s in stats.values():
        for b, z in enumerate(s["allzero"]):
            allzero_count[b] += int(z)
    print("FirePred bands checked over {} fires\n".format(len(stats)))
    print("{:>5}  {:>14}  {}".format("band", "all-zero in", "note"))
    for b in range(nb):
        note = "DROPPED in manuscript" if b == 2 else ""
        print("{:>5}  {:>10d}/{:d}  {}".format(b, int(allzero_count[b]), len(stats), note))
    json.dump({"n_fires": len(stats),
               "allzero_count": allzero_count.tolist()},
              open(os.path.join(OUT, "firepred_band_check.json"), "w"), indent=1)
    print("\nNote: manuscript uses 0-indexed band 3 = index 2 above.")
else:
    print("No FirePred rasters could be read.")


FirePred bands checked over 62 fires

 band     all-zero in  note
    0           0/62  
    1           0/62  
    2          12/62  DROPPED in manuscript
    3           0/62  
    4           0/62  
    5           0/62  
    6           0/62  
    7           0/62  
    8           0/62  
    9           0/62  
   10           0/62  
   11           0/62  
   12           0/62  
   13           0/62  
   14           5/62  
   15           0/62  
   16           0/62  
   17           0/62  
   18           0/62  

Note: manuscript uses 0-indexed band 3 = index 2 above.


## 7. Files written


In [8]:
for f in sorted(glob.glob(os.path.join(OUT, "*.csv")) +
                glob.glob(os.path.join(OUT, "*.json"))):
    print("  {:.1f} KB  {}".format(os.path.getsize(f) / 1e3, f))
print("\nDownload these and send me audit_summary_table.csv and crop_coverage.csv.")


  17.5 KB  /kaggle/working/audit_per_fire.csv
  0.3 KB  /kaggle/working/audit_summary_table.csv
  3.8 KB  /kaggle/working/crop_coverage.csv
  0.3 KB  /kaggle/working/excluded_af_train.json
  0.6 KB  /kaggle/working/excluded_ba_train.json
  0.1 KB  /kaggle/working/firepred_band_check.json

Download these and send me audit_summary_table.csv and crop_coverage.csv.
